# Week 83, Day 2: pandas I

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w083d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('orders.csv', 'w', encoding='utf-8') as f:
    f.write('order_id,date,customer,city,product,category,quantity,unit_price\n1001,2026-01-04,Ada Lovelace,York,Notebook,stationery,3,2.50\n1002,2026-01-04,Bo Chen,Leeds,Desk lamp,home,1,24.99\n1003,2026-01-11,Cy Okafor,York,Pens (10),stationery,2,4.20\n1004,2026-01-19,Dee Patel,Hull,Kettle,home,1,32.00\n1005,2026-02-02,Ada Lovelace,York,Mug,home,4,6.75\n1006,2026-02-03,Eli Novak,Bath,Notebook,stationery,10,2.50\n1007,2026-02-14,Bo Chen,Leeds,Headphones,electronics,1,59.00\n1008,2026-02-20,Fay Morgan,Bath,Charger,electronics,2,15.50\n1009,2026-03-01,Cy Okafor,York,Desk lamp,home,1,24.99\n1010,2026-03-07,Gus Adeyemi,Hull,Keyboard,electronics,1,45.00\n1011,2026-03-15,Dee Patel,Hull,Pens (10),stationery,5,4.20\n1012,2026-03-22,Ada Lovelace,York,Headphones,electronics,1,59.00\n1013,2026-03-28,Hal Brooks,Leeds,Mug,home,2,6.75\n1014,2026-04-02,Eli Novak,Bath,Charger,electronics,1,15.50\n')

**Predict before you run:** what will this print?



In [ ]:
import pandas as pd

df = pd.read_csv("orders.csv")
print(df.shape)
print(df.columns.tolist()[:4])
york = df[df["city"] == "York"]
print(len(york), york["customer"].unique().tolist())
print(round(float((df["quantity"] * df["unit_price"]).sum()), 2))

In [ ]:
import pandas as pd

prices = pd.Series([2.5, 24.99, 6.75], index=["notebook", "lamp", "mug"])
print(prices["lamp"], prices.index.tolist())
print((prices * 2).tolist())

df = pd.DataFrame({"city": ["York", "Leeds", "Hull"], "orders": [5, 3, 3]})
print(df)
print(df["orders"].sum(), type(df["orders"]).__name__)
rows = pd.DataFrame([{"city": "Bath", "orders": 3}, {"city": "Ely", "orders": 1}])
print(rows.shape)

In [ ]:
import pandas as pd

df = pd.read_csv("orders.csv")
print(df.head(3))
print(len(df), df.shape[1])
print(df.dtypes["quantity"], df.dtypes["unit_price"])
print(df["unit_price"].describe().round(2).to_dict())
print(df["category"].nunique(), sorted(df["category"].unique()))

In [ ]:
import pandas as pd

df = pd.read_csv("orders.csv")
print(df[["order_id", "city"]].head(2).values.tolist())
print(df.loc[0:2, "customer"].tolist())
print(df.iloc[-1]["product"], df.iloc[0, 2])
mask = (df["city"] == "York") & (df["quantity"] > 1)
print(df.loc[mask, "order_id"].tolist())
print(df.loc[df["city"].isin(["Hull", "Bath"]) & ~(df["category"] == "home"), "order_id"].tolist())
print(df.loc[df["quantity"].between(2, 4), "product"].tolist())

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv("orders.csv")
df["total"] = (df["quantity"] * df["unit_price"]).round(2)
df["region"] = df["city"].map({"York": "North", "Leeds": "North", "Hull": "North", "Bath": "South"})
df["size"] = np.where(df["total"] >= 30, "large", "small")
df["first_name"] = df["customer"].str.split(" ").str[0]
df["month"] = df["date"].str[:7]
df.loc[df["category"] == "electronics", "warranty"] = 2
print(df[["order_id", "total", "region", "size", "first_name", "month"]].head(3).values.tolist())
print(df["warranty"].tolist()[5:9])
lighter = df.assign(total_with_vat=(df["total"] * 1.2).round(2))
print("total_with_vat" in df.columns, lighter["total_with_vat"].iloc[0])

In [ ]:
import pandas as pd

df = pd.read_csv("orders.csv")
df["total"] = (df["quantity"] * df["unit_price"]).round(2)
print(df.sort_values("total", ascending=False)["order_id"].head(3).tolist())
print(df.sort_values(["city", "total"], ascending=[True, False])[["city", "total"]].head(3).values.tolist())
print(df.nsmallest(2, "total")["product"].tolist())
print(df["category"].value_counts().sort_index().to_dict())
print(df.loc[df["total"].idxmax(), "customer"], round(float(df["total"].mean()), 2))

> **Key idea.** A Series is a labelled 1D array; a DataFrame is a table of named columns sharing a row index. Read files with `pd.read_csv`, then inspect with `head`, `shape`, `columns`, `dtypes`, `describe` and `nunique`. Select columns with `df["c"]` or `df[["a", "b"]]`, rows by label with `.loc` (slices include the end), by position with `.iloc`, and by condition with masks combined with `&`, `|` and `~`, each condition in parentheses; `isin`, `between` and `.str.contains` build masks. Add columns with vectorised arithmetic, `map`, `np.where`, `apply` or `.str` methods; change some rows with one `df.loc[mask, "col"] = value`, never chained indexing; use `copy()` or `assign` to leave the original alone. Sort with `sort_values`, `nlargest` and `reset_index(drop=True)`; count with `value_counts`; find the row of the largest with `idxmax`. Turn results into plain Python with `tolist` and `to_dict`.

### Exercise: Load and summarise

Write `load_orders(path)`: read the CSV and add a `total` column, `quantity × unit_price` rounded to 2 places; return the DataFrame. Then `summary(df)` for a DataFrame with that column: a dict with `rows`, `customers` (distinct), `revenue` (the sum of `total`, rounded to 2 places, as a float), `first` and `last` (the earliest and latest `date`, as the strings in the file), and `categories` (the sorted list of distinct categories).

In [ ]:
import pandas as pd

def load_orders(path):
    ___

def summary(df):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_load():\n    \"read, compute, summarise\"\n    df = load_orders(\"orders.csv\")\n    assert len(df) == 14 and \"total\" in df.columns\n    assert df[\"total\"].tolist()[:3] == [7.5, 24.99, 8.4]\n    assert summary(df) == {\"rows\": 14, \"customers\": 8, \"revenue\": 393.88, \"first\": \"2026-01-04\", \"last\": \"2026-04-02\",\n                           \"categories\": [\"electronics\", \"home\", \"stationery\"]}")

### Exercise: Asking questions with masks

Using the orders (with `total`, as from `load_orders`, given), write: `big_orders(df, minimum)`: the `order_id`s of orders whose total is at least `minimum`, in file order; `in_cities(df, cities, category)`: the `order_id`s in any of `cities` **and** of `category`; `products_for(df, customer)`: the distinct products a customer bought, sorted; and `either(df, city, min_quantity)`: how many orders were in `city` **or** had a quantity of at least `min_quantity`.

In [ ]:
import pandas as pd

def load_orders(path):
    df = pd.read_csv(path)
    df["total"] = (df["quantity"] * df["unit_price"]).round(2)
    return df

def big_orders(df, minimum):
    ___

def in_cities(df, cities, category):
    ___

def products_for(df, customer):
    ___

def either(df, city, min_quantity):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_select():\n    \"& and |, in parentheses\"\n    df = load_orders(\"orders.csv\")\n    assert big_orders(df, 30) == [1004, 1007, 1008, 1010, 1012]\n    assert in_cities(df, [\"York\", \"Hull\"], \"stationery\") == [1001, 1003, 1011]\n    assert in_cities(df, [\"Ely\"], \"home\") == []\n    assert products_for(df, \"Ada Lovelace\") == [\"Headphones\", \"Mug\", \"Notebook\"]\n    assert either(df, \"Bath\", 4) == 5")

### Exercise: Derived columns

Write `enrich(df)` returning a **new** DataFrame (leave `df` unchanged) with these columns added: `total` (`quantity × unit_price`, 2 places); `band`: `"small"` for totals under 20, `"medium"` under 50, else `"large"`; `city_code`: the first three letters of the city in capitals; `surname`: the last word of `customer`; and `month`: the month number from `date` as a whole number (`"2026-03-15"` gives 3).

In [ ]:
import numpy as np
import pandas as pd

def enrich(df):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import pandas as pd\n\ndef test_enrich():\n    \"vectorised, and the original untouched\"\n    df = pd.read_csv(\"orders.csv\")\n    before = df.columns.tolist()\n    out = enrich(df)\n    assert df.columns.tolist() == before, \"leave df unchanged\"\n    assert out[\"band\"].tolist()[:8] == [\"small\", \"medium\", \"small\", \"medium\", \"medium\", \"medium\", \"large\", \"medium\"]\n    assert out[\"city_code\"].tolist()[:4] == [\"YOR\", \"LEE\", \"YOR\", \"HUL\"]\n    assert out[\"surname\"].tolist()[:3] == [\"Lovelace\", \"Chen\", \"Okafor\"]\n    assert out[\"month\"].tolist()[3:6] == [1, 2, 2] and int(out[\"month\"].max()) == 4")

### Exercise: Sorting and counting

Write `top_orders(df, n)`: the `n` orders with the largest totals as `[(order_id, total), ...]`, largest first, ties broken by the smaller `order_id` (use `sort_values` with two keys). `category_share(df)`: each category's share of the **number** of orders, as a dict of fractions rounded to 2 places. `busiest_city(df)`: the city with most orders. `total` is already in `df`.

In [ ]:
import pandas as pd

def top_orders(df, n):
    ___

def category_share(df):
    ___

def busiest_city(df):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import pandas as pd\n\ndef test_ranking():\n    \"sort_values and value_counts\"\n    df = pd.read_csv(\"orders.csv\")\n    df[\"total\"] = (df[\"quantity\"] * df[\"unit_price\"]).round(2)\n    assert top_orders(df, 4) == [(1007, 59.0), (1012, 59.0), (1010, 45.0), (1004, 32.0)]\n    assert category_share(df) == {\"electronics\": 0.36, \"home\": 0.36, \"stationery\": 0.29}\n    assert busiest_city(df) == \"York\"")

### Exercise: Tidying text

Customer names typed into a form arrive messy. Write `tidy(names)` for a Series of strings: strip spaces at both ends, collapse runs of spaces inside to one (`.str.replace(r"\s+", " ", regex=True)`), and put in title case; return the Series. Then `initials(names)`: the tidied names' initials, like `"A.L."`, as a list.

In [ ]:
import pandas as pd

def tidy(names):
    ___

def initials(names):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import pandas as pd\n\ndef test_text():\n    \"the .str methods\"\n    raw = pd.Series([\"  ada   lovelace\", \"BO CHEN \", \"cy  okafor-smith\", \"dee\"])\n    assert tidy(raw).tolist() == [\"Ada Lovelace\", \"Bo Chen\", \"Cy Okafor-Smith\", \"Dee\"]\n    assert initials(raw) == [\"A.L.\", \"B.C.\", \"C.O.\", \"D.\"]")

### Exercise: Labels and positions

After sorting, labels and positions differ. Write `label_vs_position(df)` for the orders: sort by `unit_price` from highest to lowest (ties by `order_id`), then return a tuple of: the `product` at **position** 0 (`iloc`), the `product` with **label** 0 (`loc`, which still means the first row of the file), and the list of `order_id`s of the first three rows **after** renumbering the index with `reset_index(drop=True)` and then taking labels 0 to 2 with `.loc` (remember `.loc` slices include the end).

In [ ]:
import pandas as pd

def label_vs_position(df):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import pandas as pd\n\ndef test_positions():\n    \"iloc and loc\"\n    df = pd.read_csv(\"orders.csv\")\n    assert label_vs_position(df) == (\"Headphones\", \"Notebook\", [1007, 1012, 1010])")

### Bug hunt: A discount that didn't stick

York customers should get 10% off, but every discount is still `0.0`: `[0.1, 0.0, 0.1]` is what it should print. Chained indexing changed a temporary copy. Fix it with a single `.loc`.

In [ ]:
import pandas as pd

df = pd.DataFrame({"city": ["York", "Leeds", "York"], "discount": [0.0, 0.0, 0.0]})
df[df["city"] == "York"]["discount"] = 0.1
print(df["discount"].tolist())

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"one .loc\"\n    expect_output(\"[0.1, 0.0, 0.1]\")")

### Bug hunt: The truth value of a Series

This crashes with `ValueError: The truth value of a Series is ambiguous`. Combine the conditions the pandas way so it prints `[1001, 1003]`.

In [ ]:
import pandas as pd

df = pd.read_csv("orders.csv")
picked = df[(df["city"] == "York") and (df["category"] == "stationery")]
print(picked["order_id"].tolist())

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"& with parentheses\"\n    expect_output(\"[1001, 1003]\")")

### Bug hunt: Prices glued together

The prices are text with a pound sign, so `sum` joins the strings: `£2.50£24.99£4.20`. Turn them into numbers first so it prints `31.69`.

In [ ]:
import pandas as pd

prices = pd.Series(["£2.50", "£24.99", "£4.20"])
print(prices.sum())

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"text to numbers\"\n    expect_output(\"31.69\")")

### Puzzle: The biggest York orders

Arrange the lines to load the orders, add totals, keep York's orders, and print the two biggest order ids. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
import pandas as pd

df = pd.DataFrame({"a": [5, 1, 4], "b": ["x", "y", "x"]}, index=[10, 20, 30])
p = ___   # df.loc[20, "a"]
q = ___   # df.iloc[0]["b"]
r = ___   # df.loc[10:20, "a"].tolist()
s = ___   # df.sort_values("a").index.tolist()
t = ___   # df["b"].value_counts().to_dict()

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 1 and q == \"x\" and r == [5, 1], \"p, q and r\"\n    assert s == [20, 30, 10] and t == {\"x\": 2, \"y\": 1}, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a tiny query language

Write `select(df, where=(), columns=None, order_by=None, limit=None)`, which answers questions about any DataFrame. `where` is a sequence of conditions `(column, op, value)`, all of which must hold; the ops are `"=="`, `"!="`, `"<"`, `"<="`, `">"`, `">="`, `"in"` (the value is a list), `"contains"` (text containing the value, ignoring case) and `"between"` (a pair, both ends included). An unknown column raises `KeyError("no column X")` and an unknown op `ValueError("unknown op X")`. `columns` lists the columns to keep (all if `None`, again checked). `order_by` is a column name, or the name with a leading `-` for descending, or a list of those (sorted in that order of priority). `limit` keeps the first so many rows. Return the result as a list of row dicts (`to_dict("records")`). Don't change `df`.

In [ ]:
import pandas as pd

def select(df, where=(), columns=None, order_by=None, limit=None):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import pandas as pd\n\ndef _df():\n    df = pd.read_csv(\"orders.csv\")\n    df[\"total\"] = (df[\"quantity\"] * df[\"unit_price\"]).round(2)\n    return df\n\ndef test_where():\n    \"conditions combine with and\"\n    df = _df()\n    rows = select(df, [(\"city\", \"==\", \"York\"), (\"total\", \">\", 20)], columns=[\"order_id\", \"total\"])\n    assert rows == [{\"order_id\": 1005, \"total\": 27.0}, {\"order_id\": 1009, \"total\": 24.99}, {\"order_id\": 1012, \"total\": 59.0}]\n    assert [r[\"order_id\"] for r in select(df, [(\"city\", \"in\", [\"Bath\", \"Hull\"]), (\"quantity\", \"between\", (2, 5))])] == [1008, 1011]\n    assert [r[\"product\"] for r in select(df, [(\"product\", \"contains\", \"LAMP\")], columns=[\"product\"])] == [\"Desk lamp\", \"Desk lamp\"]\n    assert len(select(df, [(\"category\", \"!=\", \"home\"), (\"unit_price\", \"<=\", 4.2)])) == 4\n    assert len(select(df)) == 14\n\ndef test_order_and_limit():\n    \"sorting and limits\"\n    df = _df()\n    rows = select(df, columns=[\"order_id\", \"city\", \"total\"], order_by=[\"city\", \"-total\"], limit=4)\n    assert [(r[\"city\"], r[\"order_id\"]) for r in rows] == [(\"Bath\", 1008), (\"Bath\", 1006), (\"Bath\", 1014), (\"Hull\", 1010)]\n    assert select(df, columns=[\"order_id\"], order_by=\"-order_id\", limit=2) == [{\"order_id\": 1014}, {\"order_id\": 1013}]\n    assert \"total\" in df.columns and len(df) == 14\n\ndef test_errors():\n    \"clear errors\"\n    df = _df()\n    for kwargs, kind, message in (({\"where\": [(\"colour\", \"==\", \"red\")]}, KeyError, \"no column colour\"),\n                                  ({\"where\": [(\"city\", \"~\", \"York\")]}, ValueError, \"unknown op ~\"),\n                                  ({\"columns\": [\"city\", \"size\"]}, KeyError, \"no column size\"),\n                                  ({\"order_by\": \"-size\"}, KeyError, \"no column size\")):\n        try:\n            select(df, **kwargs)\n        except kind as e:\n            assert message in str(e), str(e)\n        else:\n            raise AssertionError(f\"{kwargs} should fail\")")

## Warm-up

The page gives every program a copy of `orders.csv` (open it above). One line reads it into a **DataFrame**: a table whose columns have names. `df["city"] == "York"` compares a whole column at once, exactly like a NumPy mask, and `df[mask]` keeps the matching rows. That's most of pandas: named columns, and NumPy-style operations on them.

## Series and DataFrames

A **Series** is a one-dimensional array with a label for each element, its **index**. A **DataFrame** is a table: a set of Series (the columns) sharing one index (the row labels). Unless you say otherwise, the index is `0, 1, 2, ...`.

Build a DataFrame from a dict of columns, or a list of row dicts. A column is a Series; `df["name"]` gets one. Operations on Series are vectorised (NumPy underneath) and **line up by label**, not by position.

## Reading and inspecting

`pd.read_csv(path)` reads a CSV file: the header row names the columns, and pandas works out a type for each column (whole numbers, decimals, or text). Then look before doing anything:

- `df.head(n)` and `df.tail(n)`: the first or last rows;
- `df.shape`: `(rows, columns)`; `df.columns`: the names; `len(df)`: the number of rows;
- `df.dtypes`: each column's type: `int64`, `float64`, and text as `object` (or `str` in pandas 3);
- `df.describe()`: count, mean, spread and quartiles of each number column;
- `df["col"].unique()` and `.nunique()`: the distinct values and how many.

`.tolist()` and `.to_dict()` turn pandas results into plain Python, which is how the checks on this page compare answers.

## Selecting

- **Columns**: `df["city"]` is a Series; `df[["city", "total"]]` (a list inside) is a smaller DataFrame.
- **Rows by label**: `df.loc[row_labels, column_labels]`, with slices **including** both ends: `df.loc[0:2, "city"]` is rows 0, 1 and 2.
- **Rows by position**: `df.iloc[positions, positions]`, with Python-style slices: `df.iloc[0:2]` is the first two rows.
- **Rows by condition**: `df[mask]` or `df.loc[mask, columns]`. Combine conditions with `&` (and), `|` (or) and `~` (not), with **parentheses round each one**: `df[(df["city"] == "York") & (df["quantity"] > 1)]`. Python's `and` and `or` don't work on Series.
- **Helpers**: `df["city"].isin(["York", "Hull"])`, `df["quantity"].between(2, 5)` (inclusive), `df["product"].str.contains("lamp")`.

## New and changed columns

Assigning to a new column name adds a column, computed for every row at once: `df["total"] = df["quantity"] * df["unit_price"]`. Other ways to compute one:

- `df["city"].map({"York": "North", ...})`: look each value up in a dict (missing keys become `NaN`);
- `np.where(condition, if_true, if_false)`: a choice per row;
- `df["col"].apply(function)`: call a Python function per value (slower; use it when nothing vectorised fits);
- the **`.str`** methods on text columns: `.str.strip()`, `.str.lower()`, `.str.title()`, `.str.upper()`, `.str[:3]`, `.str.replace(old, new)`, `.str.split(" ").str[0]`, `.str.len()`.

To change **some** rows of a column, use one `.loc` with both parts: `df.loc[mask, "discount"] = 0.1`. Never chain two indexing steps, `df[mask]["discount"] = 0.1`: the first step makes a new table, the assignment changes that, and the original is untouched (pandas warns you, and from pandas 3 it never works). And if a function shouldn't change the DataFrame it's given, work on `df.copy()` or build a new one with **`df.assign(total=...)`**.

Rows that `.loc` didn't touch got `NaN` in the new `warranty` column: pandas fills a partly assigned column with "missing".

## Sorting and counting

- `df.sort_values("total", ascending=False)`; several keys: `sort_values(["city", "total"], ascending=[True, False])`. The index labels travel with their rows; `reset_index(drop=True)` renumbers.
- `df.nlargest(3, "total")` and `nsmallest`.
- `df["city"].value_counts()`: how often each value appears, most common first; `normalize=True` gives fractions.
- Column statistics: `sum`, `mean`, `median`, `min`, `max`, `count` (non-missing values), `idxmax` (the **label** of the largest).

## Practice

## Quiz

## Daily challenge

## Before you go

Find a CSV file you're interested in (many councils and governments publish open data: bus punctuality, air quality, library loans) or export one from a spreadsheet of your own. Read it with pandas, look at `head`, `shape` and `dtypes`, and ask it three questions with masks. Tomorrow is **pandas II**: grouping and summarising, joining tables, reshaping, dates and missing values.